# 🔄 Slowly Changing Dimensions (SCD 0, 1, 2, 3) in Pure SQL

In enterprise data warehousing, entity attributes naturally change over time (e.g. customer addresses, loyalty tiers, credit limits).
This notebook demonstrates how each SCD type is implemented in **pure SQL using real `INSERT`, `UPDATE`, and point-in-time queries**.
Each SCD type is shown with an explicit **🔴 [BEFORE]** and **🟢 [AFTER]** snapshot so you can immediately see the effect on the table.

- **SCD Type 0 (Fixed / Retain Original)**: Attributes are immutable. Incoming updates are strictly ignored.
- **SCD Type 1 (In-Place Overwrite)**: Destructive update. Existing rows are overwritten in-place with zero history.
- **SCD Type 2 (Historical Row Versioning)**: Full audit trail. Closes the active version and inserts a new version row.
- **SCD Type 3 (Current vs Previous Columns)**: Retains immediate past and present values side-by-side in adjacent columns.


In [10]:
import os
import sys
from pathlib import Path
import duckdb
import pandas as pd

# Ensure working directory is project root
while not (Path.cwd() / 'part1_tech_news_lakehouse').is_dir() and Path.cwd().parent != Path.cwd():
    os.chdir('..')

if '.' not in sys.path:
    sys.path.insert(0, '.')

# Initialize in-memory DuckDB engine
con = duckdb.connect(database=':memory:')

# Register raw CSV and create clean staging view
con.execute("""
    CREATE OR REPLACE VIEW raw_customers AS 
    SELECT * FROM read_csv_auto('data/raw/ecommerce/customers.csv');

    CREATE OR REPLACE VIEW v_staging_customers AS
    SELECT 
        TRIM(customerID) AS customer_id,
        TRIM(companyName) AS company_name,
        TRIM(contactName) AS contact_name,
        COALESCE(TRIM(city), 'Unknown') AS city,
        COALESCE(TRIM(country), 'Unknown') AS country,
        COALESCE(TRIM(region), 'N/A') AS region,
        COALESCE(TRIM(phone), 'N/A') AS phone
    FROM raw_customers;
""")

print("DuckDB connected and staging view created (91 customers)!")
display(con.execute("SELECT customer_id, company_name, city, country FROM v_staging_customers LIMIT 3").df())


DuckDB connected and staging view created (91 customers)!


,customer_id,company_name,city,country
0,ALFKI,Alfreds Futterkiste,Berlin,Germany
1,ANATR,Ana Trujillo Emparedados y helados,México D.F.,Mexico
2,ANTON,Antonio Moreno Taquería,México D.F.,Mexico


## 1. SCD Type 0: Fixed / Retain Original (Insert-Only / Immutable)

- **Concept**: Original attributes are written once during account creation and **never updated**.
- **Scenario**: Customer `BERGS` registered on `1996-01-01` in `Sweden` with an initial limit of `$5,000`.
- **Incoming Event**: An upstream system sends an update claiming the customer registered in `Norway` with a limit of `$99,999`.
- **Result**: The update is ignored (`ON CONFLICT DO NOTHING`). Original signup attributes remain 100% immutable.

In [11]:
# 1. Initialize Baseline Table with PRIMARY KEY
con.execute("""
    CREATE OR REPLACE TABLE dim_customer_scd0 (
        customer_id VARCHAR PRIMARY KEY,
        company_name VARCHAR,
        contact_name VARCHAR,
        original_country VARCHAR,
        original_signup_date DATE,
        initial_credit_limit_usd INT
    );

    INSERT INTO dim_customer_scd0
    SELECT 
        customer_id,
        company_name,
        contact_name,
        country AS original_country,
        '1996-01-01'::DATE AS original_signup_date,
        5000 AS initial_credit_limit_usd
    FROM v_staging_customers;
""")

print("[BEFORE UPDATE ATTEMPT] Customer BERGS in SCD Type 0:")
display(con.execute("SELECT * FROM dim_customer_scd0 WHERE customer_id = 'BERGS'").df())

# 2. Simulated Upstream Update Attempt (Ignored via ON CONFLICT DO NOTHING)
con.execute("""
    INSERT INTO dim_customer_scd0 (customer_id, company_name, contact_name, original_country, original_signup_date, initial_credit_limit_usd)
    VALUES ('BERGS', 'Berglunds snabbkop', 'Christina Berglund', 'Norway', '1998-05-01'::DATE, 99999)
    ON CONFLICT (customer_id) DO NOTHING;
""")

print("[AFTER UPDATE ATTEMPT] Customer BERGS in SCD Type 0 (Original values strictly preserved):")
display(con.execute("SELECT * FROM dim_customer_scd0 WHERE customer_id = 'BERGS'").df())


[BEFORE UPDATE ATTEMPT] Customer BERGS in SCD Type 0:


,customer_id,company_name,contact_name,original_country,original_signup_date,initial_credit_limit_usd
0,BERGS,Berglunds snabbköp,Christina Berglund,Sweden,1996-01-01,5000


[AFTER UPDATE ATTEMPT] Customer BERGS in SCD Type 0 (Original values strictly preserved):


,customer_id,company_name,contact_name,original_country,original_signup_date,initial_credit_limit_usd
0,BERGS,Berglunds snabbköp,Christina Berglund,Sweden,1996-01-01,5000


## 2. SCD Type 1: In-Place Overwrite (Zero History)

- **Concept**: When an attribute changes, a direct `UPDATE` statement overwrites the existing row.
- **Scenario**: Customer `BERGS` relocates from **Lulea** to **Stockholm**, and their loyalty tier upgrades from **Bronze** to **Platinum**.
- **Result**: The previous values (`Lulea`, `Bronze`) are permanently destroyed. No historical trail exists.

In [12]:
# 1. Initialize Baseline SCD Type 1 Table
con.execute("""
    CREATE OR REPLACE TABLE dim_customer_scd1 AS
    SELECT 
        customer_id,
        company_name,
        contact_name,
        city,
        country,
        'Bronze' AS loyalty_tier,
        5000 AS credit_limit_usd,
        '1996-01-01'::VARCHAR AS last_updated_at
    FROM v_staging_customers;
""")

print("[BEFORE UPDATE] Customer BERGS in SCD Type 1:")
display(con.execute("SELECT customer_id, city, loyalty_tier, credit_limit_usd, last_updated_at FROM dim_customer_scd1 WHERE customer_id = 'BERGS'").df())

# 2. Standard SQL UPDATE (In-Place Overwrite)
con.execute("""
    UPDATE dim_customer_scd1
    SET city = 'Stockholm',
        loyalty_tier = 'Platinum',
        credit_limit_usd = 50000,
        last_updated_at = '1998-04-01'
    WHERE customer_id = 'BERGS';
""")

print("[AFTER UPDATE] Customer BERGS in SCD Type 1 (Overwritten in-place, zero history):")
display(con.execute("SELECT customer_id, city, loyalty_tier, credit_limit_usd, last_updated_at FROM dim_customer_scd1 WHERE customer_id = 'BERGS'").df())


[BEFORE UPDATE] Customer BERGS in SCD Type 1:


,customer_id,city,loyalty_tier,credit_limit_usd,last_updated_at
0,BERGS,Luleå,Bronze,5000,1996-01-01


[AFTER UPDATE] Customer BERGS in SCD Type 1 (Overwritten in-place, zero history):


,customer_id,city,loyalty_tier,credit_limit_usd,last_updated_at
0,BERGS,Stockholm,Platinum,50000,1998-04-01


## 3. SCD Type 2: Historical Row Versioning (Full Audit Trail)

- **Concept**: Instead of overwriting, we track **every historical change** using a 2-step SQL mutation:
  1. `UPDATE`: Expire the active version by setting `effective_end_date` and `is_current_flag = FALSE`.
  2. `INSERT`: Add a new active version row with a new surrogate key (`customer_sk`), `effective_end_date = '9999-12-31'`, and `is_current_flag = TRUE`.
- **Scenario**: Customer `BERGS` is upgraded from **Silver** to **Platinum** on `1998-04-01`.

In [13]:
# 1. Initialize Baseline SCD Type 2 Table (Version 1 active)
con.execute("""
    CREATE OR REPLACE TABLE dim_customer_scd2 AS
    SELECT 
        'CUST_SK_' || LPAD((ROW_NUMBER() OVER (ORDER BY customer_id))::VARCHAR, 4, '0') AS customer_sk,
        customer_id,
        company_name,
        contact_name,
        city,
        country,
        'Silver' AS loyalty_tier,
        10000 AS credit_limit_usd,
        '1996-01-01'::VARCHAR AS effective_start_date,
        '9999-12-31'::VARCHAR AS effective_end_date,
        TRUE AS is_current_flag
    FROM v_staging_customers;
""")

print("[BEFORE UPDATE] Customer BERGS in SCD Type 2 (1 version active):")
display(con.execute("""
    SELECT customer_sk, customer_id, city, loyalty_tier, effective_start_date, effective_end_date, is_current_flag 
    FROM dim_customer_scd2 WHERE customer_id = 'BERGS'
""").df())

# 2. Step A: Expire active version
con.execute("""
    UPDATE dim_customer_scd2
    SET effective_end_date = '1998-03-31',
        is_current_flag = FALSE
    WHERE customer_id = 'BERGS' AND is_current_flag = TRUE;
""")

# 3. Step B: Insert new active version
con.execute("""
    INSERT INTO dim_customer_scd2 (
        customer_sk, customer_id, company_name, contact_name, city, country,
        loyalty_tier, credit_limit_usd, effective_start_date, effective_end_date, is_current_flag
    )
    VALUES (
        'CUST_SK_0092', 'BERGS', 'Berglunds snabbkop', 'Christina Berglund', 'Stockholm', 'Sweden',
        'Platinum', 50000, '1998-04-01', '9999-12-31', TRUE
    );
""")

print("[AFTER UPDATE] Customer BERGS in SCD Type 2 (Both historical and active versions preserved):")
display(con.execute("""
    SELECT customer_sk, customer_id, city, loyalty_tier, credit_limit_usd, effective_start_date, effective_end_date, is_current_flag 
    FROM dim_customer_scd2 WHERE customer_id = 'BERGS' ORDER BY effective_start_date
""").df())


[BEFORE UPDATE] Customer BERGS in SCD Type 2 (1 version active):


,customer_sk,customer_id,city,loyalty_tier,effective_start_date,effective_end_date,is_current_flag
0,CUST_SK_0005,BERGS,Luleå,Silver,1996-01-01,9999-12-31,True


[AFTER UPDATE] Customer BERGS in SCD Type 2 (Both historical and active versions preserved):


,customer_sk,customer_id,city,loyalty_tier,credit_limit_usd,effective_start_date,effective_end_date,is_current_flag
0,CUST_SK_0005,BERGS,Luleå,Silver,10000,1996-01-01,1998-03-31,False
1,CUST_SK_0092,BERGS,Stockholm,Platinum,50000,1998-04-01,9999-12-31,True


## 4. Point-in-Time Temporal Query (As-Of Date Query)

- **Concept**: Because SCD Type 2 preserves date windows (`effective_start_date` to `effective_end_date`), we can reconstruct customer state at any historical moment using standard SQL `BETWEEN`.
- **Example**: What was customer `BERGS`'s tier on **1997-06-15** vs **1998-05-01**?

In [14]:
print("=== Point-in-Time: Customer BERGS as of 1997-06-15 (Silver in Lulea) ===")
display(con.execute("""
    SELECT customer_sk, customer_id, city, loyalty_tier, credit_limit_usd, effective_start_date, effective_end_date, is_current_flag
    FROM dim_customer_scd2
    WHERE customer_id = 'BERGS'
      AND '1997-06-15' BETWEEN effective_start_date AND effective_end_date
""").df())

print("=== Point-in-Time: Customer BERGS as of 1998-05-01 (Platinum in Stockholm) ===")
display(con.execute("""
    SELECT customer_sk, customer_id, city, loyalty_tier, credit_limit_usd, effective_start_date, effective_end_date, is_current_flag
    FROM dim_customer_scd2
    WHERE customer_id = 'BERGS'
      AND '1998-05-01' BETWEEN effective_start_date AND effective_end_date
""").df())


=== Point-in-Time: Customer BERGS as of 1997-06-15 (Silver in Lulea) ===


,customer_sk,customer_id,city,loyalty_tier,credit_limit_usd,effective_start_date,effective_end_date,is_current_flag
0,CUST_SK_0005,BERGS,Luleå,Silver,10000,1996-01-01,1998-03-31,False


=== Point-in-Time: Customer BERGS as of 1998-05-01 (Platinum in Stockholm) ===


,customer_sk,customer_id,city,loyalty_tier,credit_limit_usd,effective_start_date,effective_end_date,is_current_flag
0,CUST_SK_0092,BERGS,Stockholm,Platinum,50000,1998-04-01,9999-12-31,True


## 5. SCD Type 3: Current vs Previous State Columns (Adjacent History)

- **Concept**: Instead of adding new rows, we add dedicated columns for `previous_*` values.
- **Advantage**: Zero row explosion; analytical queries can compare current vs previous directly without `LAG()` window functions.
- **Scenario**: Customer `BERGS` upgrades from **Silver** to **Platinum**.

In [15]:
# 1. Initialize Baseline SCD Type 3 Table
con.execute("""
    CREATE OR REPLACE TABLE dim_customer_scd3 AS
    SELECT 
        customer_id,
        company_name,
        contact_name,
        city,
        country,
        'Silver' AS current_loyalty_tier,
        NULL::VARCHAR AS previous_loyalty_tier,
        10000 AS current_credit_limit_usd,
        NULL::INT AS previous_credit_limit_usd,
        NULL::VARCHAR AS tier_change_date
    FROM v_staging_customers;
""")

print("[BEFORE UPDATE] Customer BERGS in SCD Type 3:")
display(con.execute("""
    SELECT customer_id, current_loyalty_tier, previous_loyalty_tier, current_credit_limit_usd, previous_credit_limit_usd, tier_change_date 
    FROM dim_customer_scd3 WHERE customer_id = 'BERGS'
""").df())

# 2. Shift Current to Previous & Update Current
con.execute("""
    UPDATE dim_customer_scd3
    SET previous_loyalty_tier = current_loyalty_tier,
        previous_credit_limit_usd = current_credit_limit_usd,
        current_loyalty_tier = 'Platinum',
        current_credit_limit_usd = 50000,
        tier_change_date = '1998-04-01'
    WHERE customer_id = 'BERGS';
""")

print("[AFTER UPDATE] Customer BERGS in SCD Type 3 (Current and Previous columns side-by-side):")
display(con.execute("""
    SELECT customer_id, current_loyalty_tier, previous_loyalty_tier, current_credit_limit_usd, previous_credit_limit_usd, tier_change_date 
    FROM dim_customer_scd3 WHERE customer_id = 'BERGS'
""").df())


[BEFORE UPDATE] Customer BERGS in SCD Type 3:


,customer_id,current_loyalty_tier,previous_loyalty_tier,current_credit_limit_usd,previous_credit_limit_usd,tier_change_date
0,BERGS,Silver,None,10000,<NA>,None


[AFTER UPDATE] Customer BERGS in SCD Type 3 (Current and Previous columns side-by-side):


,customer_id,current_loyalty_tier,previous_loyalty_tier,current_credit_limit_usd,previous_credit_limit_usd,tier_change_date
0,BERGS,Platinum,Silver,50000,10000,1998-04-01
